# Weather Impact on Bike Demand Analysis

This notebook analyzes how weather conditions and event activity affect hourly bike-sharing demand in Chicago.

Goal:
- quantify how rain and temperature influence ridership
- compare behavior across member and casual riders
- test whether local events meaningfully change demand
- create a clean analytical dataset for downstream dashboarding and reporting

In [55]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("D:/Projects/chicago-bike-demand")
RAW_DIR = BASE_DIR / "01_data" / "Raw"
PROCESSED_DIR = BASE_DIR / "01_data" / "Processed"

input_path = PROCESSED_DIR / "city_hourly_demand_weather.csv"
events_path = RAW_DIR / "dim_events.csv"
output_path = PROCESSED_DIR / "final_demand_weather_events.csv"

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2209, 11)


,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour,temperature,precipitation_mm,wind_speed
0,2025-02-28,22,1,1,0,109.84,False,False,4.4,0.0,21.4
1,2025-02-28,23,25,19,6,18.15,False,False,2.9,0.0,19.5
2,2025-03-01,0,122,69,53,8.53,True,False,1.9,0.0,18.3
3,2025-03-01,1,85,52,33,8.42,True,False,0.8,0.0,17.2
4,2025-03-01,2,46,26,20,7.60,True,False,-0.2,0.0,22.1


## Load and inspect the data

In [56]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2209 entries, 0 to 2208
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   2209 non-null   object 
 1   hour                   2209 non-null   int64  
 2   trip_count             2209 non-null   int64  
 3   member_trip_count      2209 non-null   int64  
 4   casual_trip_count      2209 non-null   int64  
 5   avg_trip_duration_min  2209 non-null   float64
 6   is_weekend             2209 non-null   bool   
 7   rush_hour              2209 non-null   bool   
 8   temperature            2209 non-null   float64
 9   precipitation_mm       2209 non-null   float64
 10  wind_speed             2209 non-null   float64
dtypes: bool(2), float64(4), int64(4), object(1)
memory usage: 159.8+ KB
None

Missing values:
date                     0
hour                     0
trip_count               0
member_trip_count        0
casual_trip_co

## Create Weather Features

### Temperature buckets

### Rain Flag

In [57]:
df["is_raining"] = df["precipitation_mm"] > 0
df[["precipitation_mm", "is_raining"]].head()

,precipitation_mm,is_raining
0,0.0,False
1,0.0,False
2,0.0,False
3,0.0,False
4,0.0,False


### Temperature Bucket Feature

In [58]:
def temp_bucket(temp):
    if temp < 5:
        return "cold"
    elif temp < 20:
        return "mild"
    return "warm"

df["temp_bucket"] = df["temperature"].apply(temp_bucket)
df[["temperature", "temp_bucket"]].head()

,temperature,temp_bucket
0,4.4,cold
1,2.9,cold
2,1.9,cold
3,0.8,cold
4,-0.2,cold


### Bad Weather Flag

In [59]:
df["bad_weather"] = (
    (df["precipitation_mm"] > 0) |
    (df["temperature"] < 5)
)

df[["precipitation_mm", "temperature", "bad_weather"]].head()

,precipitation_mm,temperature,bad_weather
0,0.0,4.4,True
1,0.0,2.9,True
2,0.0,1.9,True
3,0.0,0.8,True
4,0.0,-0.2,True


### Validate Weather Features

In [60]:
print(df[["temperature", "temp_bucket"]].head())
print("\nRain flag counts:")
print(df["is_raining"].value_counts())

print("\nTemperature bucket counts:")
print(df["temp_bucket"].value_counts())

print("\nBad weather counts:")
print(df["bad_weather"].value_counts())

   temperature temp_bucket
0          4.4        cold
1          2.9        cold
2          1.9        cold
3          0.8        cold
4         -0.2        cold

Rain flag counts:
is_raining
False    1915
True      294
Name: count, dtype: int64

Temperature bucket counts:
temp_bucket
mild    1513
cold     548
warm     148
Name: count, dtype: int64

Bad weather counts:
bad_weather
False    1429
True      780
Name: count, dtype: int64


## Analyze weather impact on demand

### Analyze Rain Impact on Demand

In [62]:
rain_demand = df.groupby("is_raining")["trip_count"].mean().round(2)
rain_demand

is_raining
False    552.45
True     377.44
Name: trip_count, dtype: float64

### Analyze Temperature Impact on Demand

In [63]:
temp_demand = df.groupby("temp_bucket")["trip_count"].mean().round(2)
temp_demand

temp_bucket
cold     288.08
mild     556.33
warm    1144.09
Name: trip_count, dtype: float64

### Member vs Casual Riders by Temperature

In [64]:
rider_mix_by_temp = (
    df.groupby("temp_bucket")[["member_trip_count", "casual_trip_count"]]
    .mean()
    .round(2)
)
rider_mix_by_temp

,member_trip_count,casual_trip_count
temp_bucket,,
cold,226.33,61.75
mild,372.34,183.99
warm,715.76,428.33


### Rain Impact on Demand

Average hourly demand drops noticeably during rainy conditions.

- No rain: ~552 average trips
- Rain: ~377 average trips

This represents roughly a 32% decline in demand, showing that precipitation is one of the strongest negative drivers of ridership.

### Temperature Impact

Demand increases substantially as temperatures rise.

- Cold: ~288 average trips
- Mild: ~556 average trips
- Warm: ~1,144 average trips

Warm conditions generate nearly 4x the demand of cold conditions, making temperature one of the strongest positive demand drivers.

### Rider Behavior by Weather

Casual riders are more sensitive to weather conditions than members.

- Casual ridership rises sharply in warm conditions
- Member ridership remains relatively more stable across weather categories

This suggests casual riders are more discretionary, while members are more likely to use bikes for routine or commute-related travel.

## Integrate Event Data

### Load Event Data

In [65]:
events = pd.read_csv("D:/Projects/chicago-bike-demand/01_data/Raw/dim_events.csv")

events["event_date"] = pd.to_datetime(events["event_date"]).dt.date

events.head()

,event_name,event_date,event_type
0,Chicago Cubs Home Game,2025-03-18,sports
1,Chicago Cubs Home Game,2025-03-19,sports
2,Chicago Cubs Home Game,2025-04-08,sports
3,Chicago Cubs Home Game,2025-04-09,sports
4,Chicago Cubs Home Game,2025-05-28,sports


### Merge Event Data with Hourly Demand Data

In [66]:
df["date"] = pd.to_datetime(df["date"]).dt.date
df[["date"]].head()

,date
0,2025-02-28
1,2025-02-28
2,2025-03-01
3,2025-03-01
4,2025-03-01


In [67]:
df = df.merge(
    events,
    left_on="date",
    right_on="event_date",
    how="left"
)

df.head()

,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour,temperature,precipitation_mm,wind_speed,is_raining,temp_bucket,bad_weather,event_name,event_date,event_type
0,2025-02-28,22,1,1,0,109.84,False,False,4.4,0.0,21.4,False,cold,True,NaN,NaN,NaN
1,2025-02-28,23,25,19,6,18.15,False,False,2.9,0.0,19.5,False,cold,True,NaN,NaN,NaN
2,2025-03-01,0,122,69,53,8.53,True,False,1.9,0.0,18.3,False,cold,True,NaN,NaN,NaN
3,2025-03-01,1,85,52,33,8.42,True,False,0.8,0.0,17.2,False,cold,True,NaN,NaN,NaN
4,2025-03-01,2,46,26,20,7.60,True,False,-0.2,0.0,22.1,False,cold,True,NaN,NaN,NaN


### Create Event-Day Flag

In [68]:
df["is_event_day"] = df["event_name"].notna()
df[["event_name", "is_event_day"]].head()

,event_name,is_event_day
0,NaN,False
1,NaN,False
2,NaN,False
3,NaN,False
4,NaN,False


### Drop Temporary Merge Columns

In [69]:
df = df.drop(columns=["event_date"])

### Validate Event Integration

In [70]:
print(df["is_event_day"].value_counts())
print(df[["event_name", "event_type", "is_event_day"]].head())

is_event_day
False    1921
True      288
Name: count, dtype: int64
  event_name event_type  is_event_day
0        NaN        NaN         False
1        NaN        NaN         False
2        NaN        NaN         False
3        NaN        NaN         False
4        NaN        NaN         False


## Analyze Event Impact on Demand

In [71]:
event_demand = df.groupby("is_event_day")["trip_count"].mean().round(2)
event_demand

is_event_day
False    533.61
True     499.51
Name: trip_count, dtype: float64

### Event Impact by Rain Conditions

In [72]:
event_rain_demand = (
    df.groupby(["is_event_day", "is_raining"])["trip_count"]
    .mean()
    .round(2)
)
event_rain_demand

is_event_day  is_raining
False         False         560.31
              True          358.35
True          False         499.64
              True          498.68
Name: trip_count, dtype: float64

### Event Impact by Temperature

In [73]:
event_temp_demand = (
    df.groupby(["is_event_day", "temp_bucket"])["trip_count"]
    .mean()
    .round(2)
)
event_temp_demand

is_event_day  temp_bucket
False         cold            281.86
              mild            562.57
              warm           1125.71
True          cold            322.88
              mild            512.91
              warm           1307.13
Name: trip_count, dtype: float64

### Member vs Casual Riders on Event Days

In [74]:
event_rider_mix = (
    df.groupby("is_event_day")[["member_trip_count", "casual_trip_count"]]
    .mean()
    .round(2)
)
event_rider_mix

,member_trip_count,casual_trip_count
is_event_day,,
False,359.19,174.42
True,358.69,140.82


## Export final dataset

In [78]:
output_path = PROCESSED_DIR / "final_demand_weather_events.csv"

df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: D:\Projects\chicago-bike-demand\01_data\Processed\final_demand_weather_events.csv


In [79]:
print(df.columns.tolist())

['date', 'hour', 'trip_count', 'member_trip_count', 'casual_trip_count', 'avg_trip_duration_min', 'is_weekend', 'rush_hour', 'temperature', 'precipitation_mm', 'wind_speed', 'is_raining', 'temp_bucket', 'bad_weather', 'event_name', 'event_type', 'is_event_day']


In [80]:
print("Columns:")
print(df.columns.tolist())

print("\nPreview:")
print(df.head())

print("\nShape:")
print(df.shape)

Columns:
['date', 'hour', 'trip_count', 'member_trip_count', 'casual_trip_count', 'avg_trip_duration_min', 'is_weekend', 'rush_hour', 'temperature', 'precipitation_mm', 'wind_speed', 'is_raining', 'temp_bucket', 'bad_weather', 'event_name', 'event_type', 'is_event_day']

Preview:
         date  hour  trip_count  member_trip_count  casual_trip_count  \
0  2025-02-28    22           1                  1                  0   
1  2025-02-28    23          25                 19                  6   
2  2025-03-01     0         122                 69                 53   
3  2025-03-01     1          85                 52                 33   
4  2025-03-01     2          46                 26                 20   

   avg_trip_duration_min  is_weekend  rush_hour  temperature  \
0                 109.84       False      False          4.4   
1                  18.15       False      False          2.9   
2                   8.53        True      False          1.9   
3                   8.42

In [84]:
def save_dataframe(df, folder_path, file_name):
    folder_path.mkdir(parents=True, exist_ok=True)
    
    output_path = folder_path / file_name
    df.to_csv(output_path, index=False)
    
    print(f"File saved to: {output_path}")

In [85]:
# =========================
# Save Final Dataset
# =========================

save_dataframe(df, PROCESSED_DIR, "final_demand_weather_events.csv")

File saved to: D:\Projects\chicago-bike-demand\01_data\Processed\final_demand_weather_events.csv


## Key Takeaways

This analysis showed that external conditions play a major role in bike demand:

- Rain meaningfully reduces hourly ridership
- Warmer temperatures drive substantially higher usage
- Casual riders are more weather-sensitive than members
- Events alone have limited impact, but demand can increase when events align with favorable weather

The final cleaned dataset produced here can be used for downstream dashboarding, forecasting, or deeper segmentation analysis.